# phys119-functions: verify a system-wide install

Run this **from a regular student account**, not from an admin account, in a
folder that does not contain a copy of `phys119_functions.py`. Run every cell
in order. The last cell prints a single pass or fail verdict.

Expected version: `1.0.0`. Change `EXPECTED` in the first cell when
verifying a later release.

In [ ]:
EXPECTED = "1.0.0"

import os
import site
import sys

import phys119_functions as pf

module_path = os.path.realpath(pf.__file__)
shared_prefix = os.path.realpath(sys.prefix)
user_site = os.path.realpath(site.getusersitepackages())

print("Kernel executable:", sys.executable)
print("Shared environment:", shared_prefix)
print("Module loaded from:", module_path)
print("Version:", getattr(pf, "__version__", "NONE"))
print()

problems = []

if getattr(pf, "__version__", None) != EXPECTED:
    problems.append(
        "Version is " + str(getattr(pf, "__version__", "missing"))
        + ", expected " + EXPECTED
    )

if module_path.startswith(user_site):
    problems.append(
        "Loaded from this account's personal packages, which is hiding the "
        "shared install. Run: pip uninstall --user phys119-functions"
    )
elif module_path.startswith(shared_prefix):
    print("OK: loaded from the shared environment.")
elif os.path.dirname(module_path) == os.path.realpath(os.getcwd()):
    problems.append(
        "Loaded from the working directory. A stray phys119_functions.py "
        "sits next to this notebook."
    )
else:
    problems.append("Loaded from outside the shared environment: " + module_path)

## Values

Checked against numpy, so this catches a broken install rather than a
mistyped constant.

In [ ]:
import numpy as np

from phys119_functions import *

sample = [10.1, 10.3, 9.8]

checks = [
    ("t_score(10, 2, 12, 3)", t_score(10, 2, 12, 3), 0.5547001962252291),
    ("standard_deviation(sample)", standard_deviation(sample), float(np.std(sample, ddof=1))),
    ("standard_unc_of_mean(sample)", standard_unc_of_mean(sample),
     float(np.std(sample, ddof=1) / np.sqrt(len(sample)))),
]

for label, got, want in checks:
    ok = got is not None and abs(got - want) < 1e-12 and type(got) is float
    print(("OK  " if ok else "FAIL"), label, "->", got)
    if not ok:
        problems.append(label + " gave " + repr(got) + ", expected " + repr(want))

## What `import *` brings in

Expect exactly the three function names, and nothing else.

In [ ]:
namespace = {}
exec("from phys119_functions import *", namespace)
exported = sorted(name for name in namespace if not name.startswith("__"))

print(exported)
if exported != ["standard_deviation", "standard_unc_of_mean", "t_score"]:
    problems.append("import * exported " + repr(exported))

## Error behaviour

The next cell should print a red message about needing at least 2
measurements, with **no traceback**, and return nothing.

In [ ]:
standard_deviation([4.2])

## Verdict

In [ ]:
print()
if problems:
    print("FAILED")
    for problem in problems:
        print(" -", problem)
else:
    print("ALL CHECKS PASSED")
    print("phys119-functions", EXPECTED, "is installed for all users and behaves correctly.")